# Technique 3

## Case 4: GTO to EML1-Halo


### Test settings

Convergence tests and computational time use separate cells. Each timing test uses seven serial `%%timeit` repeats with the integrators initialized beforehand.


In [1]:
import os
import sys
import platform

for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

import numpy as np
import scipy
import heyoka as hy
from IPython.display import display, Markdown

hy.set_nthreads(1)


In [2]:
# =========================
# Timing Settings
# =========================
from collections import Counter

cpu_samples = []
fast_cpus = []
if sys.platform == 'darwin':
    import ctypes
    import plistlib
    import subprocess
    tree = plistlib.loads(subprocess.check_output(
        ['/usr/sbin/ioreg', '-a', '-l', '-p', 'IODeviceTree', '-n', 'cpus', '-r']))
    pending = list(tree)
    while pending:
        node = pending.pop()
        if node.get('cluster-type') == b'P\x00' and 'cpu-id' in node:
            fast_cpus.append(int.from_bytes(node['cpu-id'], 'little'))
        pending.extend(node.get('IORegistryEntryChildren', []))
    lib = ctypes.CDLL('/usr/lib/libSystem.B.dylib')
    lib.pthread_set_qos_class_self_np.argtypes = [ctypes.c_uint, ctypes.c_int]
    lib.pthread_cpu_number_np.argtypes = [ctypes.POINTER(ctypes.c_size_t)]

def set_timing_priority():
    if sys.platform == 'darwin':
        assert lib.pthread_set_qos_class_self_np(0x21, 0) == 0

def sample_cpu():
    if sys.platform == 'darwin':
        cpu = ctypes.c_size_t()
        assert lib.pthread_cpu_number_np(ctypes.byref(cpu)) == 0
        cpu_samples.append(cpu.value)

environment = dict(python=sys.version, executable=sys.executable,
                   platform=platform.platform(), machine=platform.machine(),
                   numpy=np.__version__, scipy=scipy.__version__, heyoka=hy.__version__,
                   threads=1, fast_cpus=sorted(fast_cpus), strict_cpu_pinning=False)
print(environment)

records_by_method = {}
results_by_method = {}

def save_trial(i, result, accepted, nfev):
    records.append(dict(trial=int(i), accepted=bool(accepted), nfev=int(nfev),
        status=int(result.status) if result is not None else None,
        residual_norm=float(np.linalg.norm(result.fun)) if result is not None else None,
        x=result.x.tolist() if result is not None else None))


{'python': '3.12.14 (main, Sep  3 2026, 03:46:19) [Clang 21.1.8 ]', 'executable': '/Users/zhong/Documents/Codex/miniforge3/envs/gtoc13_heyoka7121_paper/bin/python', 'platform': 'macOS-26.5.1-arm64-arm-64bit', 'machine': 'arm64', 'numpy': '2.5.3', 'scipy': '1.18.0', 'heyoka': '7.12.1', 'threads': 1, 'fast_cpus': [12, 13, 14, 15, 16, 17], 'strict_cpu_pinning': False}


In [3]:
import numpy as np
import heyoka as hy

# =========================
# Setup (SI Units and Nondimensionalized Units)
# =========================
day2s = 86400.0     # JPL

x0 = [ # LU, VU
    -0.019488511458668, -0.016033479812051, 0.0,
    8.918881923678198, -4.081793688818725, 0.0,
]
xf = [ # LU, VU
    0.823385182067467, 0.0, -0.022277556273235,
    0.0, 0.134184170262437, 0.0,
]

tof = 8.0*day2s # [s]

m0 = 1500.0 # [kg]
Tmax = 10.0 # [N]
Isp = 3000  # [s]

mu = 1.21506683*1e-2    # [NU]
# g0 = 9.80665            # [m∕s2]
g0 = 9.8065            # [m∕s2]

# =========================
# Nondimensionalize
# =========================
# Nondimension units
LUnit = 3.84405000*1e5*1e3
TUnit = 3.75676967*1e5
MUnit = m0

VUnit = LUnit / TUnit # VUnit to m/s
AUnit = VUnit / TUnit # AUnit to m/s^2

# x0_nu, xf_nu, t0_nu, tf_nu, m0_nu, mu_nu, c1=Tmax_nu, c2=Isp_nu*g0_nu
x0_nu = x0
xf_nu = xf
t0_nu = 0.0
tf_nu = t0_nu+tof
m0_nu = m0
mu_nu = mu
Tmax_nu = Tmax
Isp_nu = Isp
g0_nu = g0

t0_nu = t0_nu / TUnit
tf_nu = tf_nu / TUnit
m0_nu = m0_nu / MUnit
Tmax_nu = Tmax_nu / MUnit / AUnit
Isp_nu = Isp_nu / TUnit
g0_nu = g0_nu / AUnit

c1 = Tmax_nu
c2 = Isp_nu*g0_nu

# =========================
# reference value for Tmax or m0
# =========================
# Tmax_ref = 8.140347  # [N], for reference
# m0_ref = 1842.673245 # [kg], for reference
Tmax_ref = Tmax        # [N], for reference
m0_ref = m0            # [kg], for reference
Tmax_ref_nu = Tmax_ref / MUnit / AUnit
m0_ref_nu = m0_ref / MUnit


In [4]:
# =========================
# Set parameters for minpack
# =========================
import time
from scipy.optimize import root
repeat_num_minpack  = 5000

minpack_tol     = 1.0e-10
minpack_xtol    = 1.0e-10
minpack_maxfev  = 50000
minpack_factor  = 0.1
minpack_diag    = [1.0]*3 + [1.0]*3 + [1.0]*1
minpack_eps     = 0.
minpack_error   = 1.0e-7

np.set_printoptions(precision=16, suppress=True)


In [5]:
# =========================
# Define the shooting function
# =========================
def shooting_minT(lam, out = False, Time_out = None):
    # lam[0:7] : initial co-states [lx,ly,lz,lvx,lvy,lvz,lm]
    # lam[7] : nondimensional Tmax
    # function uses global variables ta(mu_nu, c1, c2), t0_nu, x0_nu, m0_nu, tf_nu, xf_nu, 
    
    ta.time = t0_nu
    # Shooting variable Tmax
    Tmax_shooting = float(lam[7])

    # Numerical protection: require positive thrust and
    # keep the full-thrust mass positive over the propagation interval.
    dt_guard = float(tf_nu - t0_nu)
    Tmax_mass_limit = np.inf if dt_guard <= 0.0 else float(m0_nu * c2 / dt_guard)
    valid_Tmax = (
        np.isfinite(Tmax_shooting)
        and (Tmax_shooting > 0.0)
        and (Tmax_shooting < Tmax_mass_limit)
    )
    if not valid_Tmax:
        if Time_out is not None:
            return None
        target = float(Tmax_nu)
        if (not np.isfinite(target)) or target <= 0.0 or target >= Tmax_mass_limit:
            target = 0.5 * Tmax_mass_limit if np.isfinite(Tmax_mass_limit) else 1.0
        scale = max(abs(target), 1.0e-12)
        penalty = np.zeros(8, dtype=float)
        penalty[-1] = ((Tmax_shooting - target) / scale) if np.isfinite(Tmax_shooting) else -1.0
        return penalty
    ta.pars[:] = [mu_nu, Tmax_shooting, c2]

    ta.state[:7] = x0_nu + [m0_nu]
    ta.state[7:14] = lam[0:7]
    ta.state[14] = 0.0 # q(t0) = 0

    if Time_out is None: # for root function
        ta.propagate_until(tf_nu)

        # Construct 8 equations: 6 terminal errors + 1 terminal co-state lm = 0 + q(tf) = 1
        retval = [
            ta.state[0] - xf_nu[0],
            ta.state[1] - xf_nu[1],
            ta.state[2] - xf_nu[2],
            ta.state[3] - xf_nu[3],
            ta.state[4] - xf_nu[4],
            ta.state[5] - xf_nu[5],
            ta.state[13],
            ta.state[14] - 1.0,
        ]

        if out:
            print("  Final Mass:", ta.state[6]*MUnit, "  Tmax:", Tmax_shooting*MUnit*AUnit)

    if Time_out is not None:
        ta.propagate_until(Time_out)

        state_t = ta.state[0:14]
        rho_t  = float(rho_func(state_t, pars=ta.pars)[0])
        rho_fuel_t  = float(rho_fuel_func(state_t, pars=ta.pars)[0])

        return {
            "t":  Time_out,
            "SG": rho_t,
            "SG_fuel": rho_fuel_t
        }

    return retval


In [6]:
# =========================
# Define the shooting function
# =========================
def shooting_maxm0(lam, out = False, Time_out = None):
    # lam[0:6] : initial co-states [lx,ly,lz,lvx,lvy,lvz]
    # lam[6] : nondimensional initial mass
    # function uses global variables ta(mu_nu, c1, c2), t0_nu, x0_nu, m0_nu, tf_nu, xf_nu, 
    
    ta.time = t0_nu
    ta.pars[:] = [mu_nu, c1, c2]

    # Shooting variable m0
    m0_shooting = float(lam[6])

    # Numerical protection: require positive initial mass and
    # keep mass positive over the full-thrust propagation interval.
    dt_guard = float(tf_nu - t0_nu)
    m0_mass_floor = max(0.0, float((c1 / c2) * dt_guard)) if c2 > 0.0 else np.inf
    valid_m0 = np.isfinite(m0_shooting) and (m0_shooting > m0_mass_floor)
    if not valid_m0:
        if Time_out is not None:
            return None
        target = float(m0_nu)
        if (not np.isfinite(target)) or target <= m0_mass_floor:
            target = max(1.0, 1.1 * m0_mass_floor)
        scale = max(abs(target), 1.0e-12)
        penalty = np.zeros(7, dtype=float)
        penalty[-1] = ((m0_shooting - target) / scale) if np.isfinite(m0_shooting) else -1.0
        return penalty
    ta.state[:7] = x0_nu + [m0_shooting]
    ta.state[7:13] = lam[0:6]
    ta.state[13] = 1.0

    if Time_out is None: # for root function
        ta.propagate_until(tf_nu)

        # Construct 7 equations: 6 terminal Cartesian state errors + 1 terminal co-state lm = 0
        retval = [
            ta.state[0] - xf_nu[0],
            ta.state[1] - xf_nu[1],
            ta.state[2] - xf_nu[2],
            ta.state[3] - xf_nu[3],
            ta.state[4] - xf_nu[4],
            ta.state[5] - xf_nu[5],
            ta.state[13],
        ]

        if out:
            print("  Final Mass:", ta.state[6]*MUnit, "  Initial Mass:", m0_shooting*MUnit)

    if Time_out is not None:
        ta.propagate_until(Time_out)

        state_t = ta.state[0:14]
        rho_t  = float(rho_func(state_t, pars=ta.pars)[0])
        rho_fuel_t  = float(rho_fuel_func(state_t, pars=ta.pars)[0])

        return {
            "t":  Time_out,
            "SG": rho_t,
            "SG_fuel": rho_fuel_t
        }

    return retval


## Method 1: Minimize the maximal thrust magnitude (with gradient)


In [7]:
import cr3bp_hamilton_minT

ta = cr3bp_hamilton_minT.ta
ta_var = cr3bp_hamilton_minT.ta_var
rho_func = cr3bp_hamilton_minT.rho_func
rho_fuel_func = cr3bp_hamilton_minT.rho_fuel_func
ta.pars[:] = [mu_nu, c1, c2]
ta_var.pars[:] = ta.pars
# =========================
# Set parameters
# =========================
# Explanatory integer-like scales for the initial shooting variables
# x = [lambda_r0(3), lambda_v0(3), lambda_m0, Tmax]
scale_lr = 1.0
scale_lv = 1.0 / 300.0
scale_lm = 1.0
# Tmax_guess is sampled from [Tmax_nu / c_T, Tmax_nu * c_T]
c_T = 2.0

# Random boxes are directly the scales
random_lr_box = scale_lr
random_lv_box = scale_lv
random_lm_box = scale_lm
scale_eta = 1.0

# MINPACK variable scaling:

# [lambda_r0(3), lambda_v0(3), lambda_m0, Tmax]
diag_cr3bp_minT = [
    1.0 / scale_lr, 1.0 / scale_lr, 1.0 / scale_lr,
    1.0 / scale_lv, 1.0 / scale_lv, 1.0 / scale_lv,
    1.0 / scale_lm,
    1.0 / scale_eta,
]

# =========================
# Define the sensitivity matrix ∂Yf/∂[lambda0, Tmax]
# =========================
def stm_15x8_minT(x0_15, Tmax, t0 = t0_nu, tf = tf_nu ):
    """
    Return Phi(tf) with shape (15, 8),
    w.r.t. [lx0, ly0, lz0, lvx0, lvy0, lvz0, lm0, Tmax].

    Here
        Y = [x, y, z, vx, vy, vz, m,
             lx, ly, lz, lvx, lvy, lvz, lm, q]
    so Phi(tf) = dY(tf) / d[lambda0, Tmax].
    """
    ta_var.pars[:] = [mu_nu, Tmax, c2]
    ta_var.time = t0

    # Initial first-order sensitivity matrix:
    
    # dY0 / d[lambda0, Tmax] =
    # [ 0_(7x7)   0_(7x1) ]
    # [ I_(7x7)   0_(7x1) ]
    # [ 0_(1x7)   0_(1x1) ]
    
    phi0 = np.zeros((15, 8), dtype=float)
    phi0[7:14, 0:7] = np.eye(7)

    ta_var.state[:] = list(x0_15) + phi0.ravel().tolist()

    # Propagate
    ta_var.propagate_until(tf)

    # Extract first-order variational block
    sl = ta_var.get_vslice(order=1)
    Phi_tf = np.asarray(ta_var.state[sl]).reshape(15, 8)

    return Phi_tf

def shooting_jac_minT(lam):
    """
    Return the shooting Jacobian J with shape (8, 8),
    w.r.t. [lx0, ly0, lz0, lvx0, lvy0, lvz0, lm0, Tmax].
    """
    lam = np.asarray(lam, dtype=float)
    
    # Direct Tmax shooting variable.
    Tmax_shooting = float(lam[7])

    # Same admissibility guard as shooting_minT().
    dt_guard = float(tf_nu - t0_nu)
    Tmax_mass_limit = np.inf if dt_guard <= 0.0 else float(m0_nu * c2 / dt_guard)
    valid_Tmax = (
        np.isfinite(Tmax_shooting)
        and (Tmax_shooting > 0.0)
        and (Tmax_shooting < Tmax_mass_limit)
    )
    if not valid_Tmax:
        target = float(Tmax_nu)
        if (not np.isfinite(target)) or target <= 0.0 or target >= Tmax_mass_limit:
            target = 0.5 * Tmax_mass_limit if np.isfinite(Tmax_mass_limit) else 1.0
        scale = max(abs(target), 1.0e-12)
        jac = np.zeros((8, 8), dtype=float)
        jac[-1, -1] = 1.0 / scale
        return jac

    x0_15 = np.asarray(
        list(x0_nu) + [m0_nu] + lam[0:7].tolist() + [0.0],
        dtype=float
    )

    phi = stm_15x8_minT(x0_15, Tmax_shooting)

    jac = np.zeros((8, 8), dtype=float)
    jac[0:6, :] = phi[0:6, :]
    jac[6, :]   = phi[13, :]
    jac[7, :]   = phi[14, :]

    # Last column of phi: derivative with respect to Tmax.

    return jac

# =========================
# Initial guess generation
# =========================
def build_initial_guess_minT():
    """
    Build a random initial shooting guess for
    [lx0, ly0, lz0, lvx0, lvy0, lvz0, lm0, Tmax].
    """
    x0 = np.zeros(8, dtype=float)

    # Random initial guess for lambda_r
    x0[0:3] = -random_lr_box + 2.0 * random_lr_box * np.random.random(3) # [−random_lr_box, random_lr_box]

    # Random initial guess for lambda_v
    x0[3:6] = -random_lv_box + 2.0 * random_lv_box * np.random.random(3) # [−random_lv_box, random_lv_box]
    while np.linalg.norm(x0[3:6]) < 1.0e-8:
        x0[3:6] = -random_lv_box + 2.0 * random_lv_box * np.random.random(3)

    # Random initial guess for lambda_m > 0
    x0[6] = random_lm_box * np.random.random() # [0, random_lm_box]

    # Log-uniform initial guess for Tmax
    
    u_T = np.random.random()
    x0[7] = Tmax_ref_nu * (c_T ** (2.0 * u_T - 1.0))
    # x0[7] = Tmax_ref_nu  # Use reference Tmax as initial guess 

    return x0

# =========================
# Run root() once for a given initial guess
# =========================
def run_root_once(lam_guess, use_jac = False):
    """
    Run scipy.optimize.root once for a given initial guess.
    lam_guess : array_like, shape (8,)  Initial guess for [lx0, ly0, lz0, lvx0, lvy0, lvz0, lm0, Tmax]. use_jac : bool, optional. If True, provide shooting_jac_minT as Jacobian.
    """
    lam_guess = np.asarray(lam_guess, dtype=float)

    root_kwargs = dict(
        fun = shooting_minT,
        x0 = lam_guess,
        method = 'hybr',
        tol = minpack_tol,
        options = {
            'xtol':   minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag':   diag_cr3bp_minT,
            # 'diag':   minpack_diag+[1.],
            'eps':    minpack_eps,
        }
    )

    if use_jac:
        root_kwargs['jac'] = shooting_jac_minT

    r = root(**root_kwargs)
    return r

# =========================
# Check whether one root() result is acceptable
# =========================
def check_root_result(r):
    if r is None:
        return False, np.inf

    res = np.linalg.norm(r.fun)

    ok = (
        bool(r.success)
        and np.isfinite(res)
        and (res < minpack_error)
    )

    return ok, res


In [8]:
# =========================
# Initialize solving statistics and best solution
# =========================
# Optional: choose whether to provide Jacobian
use_shooting_jac = True

np.random.seed(322)

count_solved   = 0
total_time     = 0.0
total_nfev     = 0
converge_nfev  = 0

global_lambda  = None
global_Tmax    = np.inf

# Record all converged solutions
all_solutions = []

# =========================
# Random initial guess loop
# =========================

# =========================
# Common random initial guess pool
# =========================
np.random.seed(322)
initial_guess_pool = []
for i in range(repeat_num_minpack):
    lam0 = build_initial_guess_minT()
    initial_guess_pool.append(lam0.copy())
initial_guess_pool = np.asarray(initial_guess_pool)


In [9]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None

    # Random initial guess
    lam0 = initial_guess_pool[i].copy()

    time_start = time.perf_counter()
    r = run_root_once(lam0, use_jac = use_shooting_jac)
    time_end = time.perf_counter()

    elapsed = time_end - time_start
    total_time += elapsed
    total_nfev += getattr(r, 'nfev', 0)

    ok, res = check_root_result(r)

    if ok:
        count_solved += 1
        converge_nfev += r.nfev

        Tmax_nu_sol = float(r.x[-1])  # direct Tmax_nu
        Tmax_si_sol = Tmax_nu_sol * MUnit * AUnit

        print(f"counter: {count_solved}/{i+1}  nfev: {r.nfev}  res: {res:.3e}", end='', flush=True)

        # Print one more time with out=True
        shooting_minT(r.x, out = True, Time_out = None)

        # Save all converged solutions
        all_solutions.append({
            "trial_index": i,
            "x_init": lam0.copy(),
            "x_sol":  r.x.copy(),
            # Last shooting variable is Tmax_nu directly.
            # "Tmax_nu": Tmax_nu_sol,
            "Tmax_SI": Tmax_si_sol,
            "res":    float(res),
            "nfev":   int(r.nfev),
            "time_s": float(elapsed),
        })

        # Save the best solution using Tmax
        if Tmax_si_sol < global_Tmax:
            global_Tmax = Tmax_si_sol
            global_lambda = r.x.copy()
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence statistics
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), optimal Tmax: {global_Tmax:.6f} N")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)
    print(f"Number of stored converged solutions: {len(all_solutions)}")
records_by_method['MinT_AJ'] = records


counter: 1/1  nfev: 112  res: 6.446e-11

  Final Mass: 1254.9101336485014   Tmax: 10.431743812395663


counter: 2/4  nfev: 64  res: 1.072e-11

  Final Mass: 1272.6953826530337   Tmax: 9.674751432348206


counter: 3/6  nfev: 61  res: 4.021e-13

  Final Mass: 1308.7456282860167   Tmax: 8.140347205786417


... (2,393 outputs left out)

counter: 1321/4992  nfev: 94  res: 7.616e-12

  Final Mass: 1272.6953826530073   Tmax: 9.67475143234931
counter: 1322/4993  nfev: 52  res: 1.718e-12

  Final Mass: 1308.7456282860176   Tmax: 8.140347205786316


Succeed to solve the problem!
Solved: 1322 in 5000 (0.2644), optimal Tmax: 8.140347 N
Average number of function calls for Minpack: 57
Average number of function calls for convergent solution: 85
global_lambda: [ 8.572995802513399  -7.101325125472983  -1.4683705067282447
 -0.0061444544115747 -0.0344873412958897 -0.000499846523852
  1.9924699491683506  1.9924699491690383]
Number of stored converged solutions: 1322


In [10]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(200)
accepted = np.array([r['accepted'] for r in records_by_method['MinT_AJ']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [11]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()

    # Random initial guess
    lam0 = initial_guess_pool[i].copy()

    r = run_root_once(lam0, use_jac = use_shooting_jac)


1min 29s ± 958 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()

    # Random initial guess
    lam0 = initial_guess_pool[i].copy()

    r = run_root_once(lam0, use_jac = use_shooting_jac)


10.1 s ± 226 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()

    # Random initial guess
    lam0 = initial_guess_pool[i].copy()

    r = run_root_once(lam0, use_jac = use_shooting_jac)


1min 21s ± 537 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['MinT_AJ'] = dict(
    method='MINT, AJ', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['MinT_AJ'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['MinT_AJ']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Method 2: Maximize the initial mass (with gradient)


In [7]:
import cr3bp_hamilton_maxm0

ta = cr3bp_hamilton_maxm0.ta
ta_var = cr3bp_hamilton_maxm0.ta_var
rho_func = cr3bp_hamilton_maxm0.rho_func
rho_fuel_func = cr3bp_hamilton_maxm0.rho_fuel_func
ta.pars[:] = [mu_nu, c1, c2]
ta_var.pars[:] = ta.pars
# =========================
# Set parameters
# =========================
# Explanatory integer-like scales for the initial shooting variables
# x = [lambda_r0(3), lambda_v0(3), m0].
scale_lr = 1.0
scale_lv = 1.0 / 300.0
# m0_guess is sampled from [m0_nu / c_m, m0_nu * c_m].
c_m = 2.0

# Random boxes are directly the scales
random_lr_box = scale_lr
random_lv_box = scale_lv
scale_eta = 1.0

# MINPACK variable scaling:

# [lambda_r0(3), lambda_v0(3), m0]
diag_cr3bp_maxm0 = [
    1.0 / scale_lr, 1.0 / scale_lr, 1.0 / scale_lr,
    1.0 / scale_lv, 1.0 / scale_lv, 1.0 / scale_lv,
    1.0 / scale_eta,
]

# =========================
# Define the sensitivity matrix ∂Yf/∂[m0, lambda_r0, lambda_v0]
# =========================
def stm_14x7_maxm0(x0_14, t0=t0_nu, tf=tf_nu):
    """
    Return Phi(tf) with shape (14, 7),
    w.r.t. [m0, lx0, ly0, lz0, lvx0, lvy0, lvz0].

    Here
        Y = [x, y, z, vx, vy, vz, m,
             lx, ly, lz, lvx, lvy, lvz, lm]

    so Phi(tf) = dY(tf) / d[m0, lambda_r0, lambda_v0].
    """
    ta_var.pars[:] = [mu_nu, c1, c2]
    ta_var.time = t0

    # Initial first-order sensitivity matrix:
    
    # dY0 / d[m0, lambda_r0, lambda_v0] =
    # [ 0_(6x1)   0_(6x6) ]
    # [ 1         0_(1x6) ]   <- m(t0) = m0
    # [ 0_(6x1)   I_(6x6) ]   <- lambda_r0, lambda_v0
    # [ 0         0_(1x6) ]   <- lambda_m(t0) = 1 is fixed
    
    phi0 = np.zeros((14, 7), dtype=float)
    phi0[6, 0] = 1.0
    phi0[7:13, 1:7] = np.eye(6)

    ta_var.state[:] = list(x0_14) + phi0.ravel().tolist()

    # Propagate
    ta_var.propagate_until(tf)

    # Extract first-order variational block
    sl = ta_var.get_vslice(order=1)
    Phi_tf = np.asarray(ta_var.state[sl]).reshape(14, 7)

    return Phi_tf

def shooting_jac_maxm0(lam):
    """
    Return the shooting Jacobian J with shape (7, 7),
    w.r.t. [lx0, ly0, lz0, lvx0, lvy0, lvz0, m0].
    
    Internally, ta_var gives sensitivities w.r.t.
    [m0, lx0, ly0, lz0, lvx0, lvy0, lvz0], so only column reordering is needed.
    """
    lam = np.asarray(lam, dtype=float)

    # Direct m0 shooting variable.
    m0_shooting = float(lam[6])

    # Same admissibility guard as shooting_maxm0().
    dt_guard = float(tf_nu - t0_nu)
    m0_mass_floor = max(0.0, float((c1 / c2) * dt_guard)) if c2 > 0.0 else np.inf
    valid_m0 = np.isfinite(m0_shooting) and (m0_shooting > m0_mass_floor)
    if not valid_m0:
        target = float(m0_nu)
        if (not np.isfinite(target)) or target <= m0_mass_floor:
            target = max(1.0, 1.1 * m0_mass_floor)
        scale = max(abs(target), 1.0e-12)
        jac = np.zeros((7, 7), dtype=float)
        jac[-1, -1] = 1.0 / scale
        return jac

    x0_14 = np.asarray(
        list(x0_nu) + [m0_shooting] + lam[0:6].tolist() + [1.0],
        dtype=float
    )

    phi = stm_14x7_maxm0(x0_14)

    residual_rows = [0, 1, 2, 3, 4, 5, 13]

    jac = np.zeros((7, 7), dtype=float)

    # Columns 0:6 correspond to [lambda_r0, lambda_v0],
    # i.e. columns 1:7 in phi.
    jac[:, 0:6] = phi[residual_rows, 1:7]

    # Last column is directly d(residual)/dm0.
    jac[:, 6] = phi[residual_rows, 0]

    return jac

# =========================
# Initial guess generation
# =========================
def build_initial_guess_maxm0():
    """
    Build a random initial shooting guess for
    [lx0, ly0, lz0, lvx0, lvy0, lvz0, m0].
    """
    x0 = np.zeros(7, dtype=float)

    # Random initial guess for lambda_r
    x0[0:3] = (-random_lr_box + 2.0 * random_lr_box * np.random.random(3))  # [-random_lr_box_maxm0, random_lr_box_maxm0]

    # Random initial guess for lambda_v
    x0[3:6] = (-random_lv_box + 2.0 * random_lv_box * np.random.random(3))  # [-random_lv_box_maxm0, random_lv_box_maxm0]

    # Avoid the undefined thrust direction at lambda_v = 0.
    while np.linalg.norm(x0[3:6]) < 1.0e-8:
        x0[3:6] = (-random_lv_box + 2.0 * random_lv_box * np.random.random(3))

    # Draw for the common random sequence used by MINT and MIM
    _ = np.random.random()

    # Log-uniform initial guess for m0
    
    u_m = np.random.random()
    x0[6] = m0_ref_nu * (c_m ** (2.0 * u_m - 1.0))
    # x0[6] = m0_ref_nu  # Use reference m0 as initial guess 

    return x0

# =========================
# Run root() once for a given initial guess
# =========================
def run_root_once_maxm0(lam_guess, use_jac=False):
    """
    Run scipy.optimize.root once for a given initial guess.
    lam_guess : array_like, shape (7,)
        Initial guess for [lx0, ly0, lz0, lvx0, lvy0, lvz0, m0]. use_jac : bool, optional
        If True, provide shooting_jac_maxm0 as Jacobian.
    """
    lam_guess = np.asarray(lam_guess, dtype=float)

    root_kwargs = dict(
        fun=shooting_maxm0,
        x0=lam_guess,
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol':   minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag':   diag_cr3bp_maxm0,
            # 'diag': minpack_diag,
            'eps':    minpack_eps,
        }
    )

    if use_jac:
        root_kwargs['jac'] = shooting_jac_maxm0

    r = root(**root_kwargs)
    return r

# =========================
# Check whether one root() result is acceptable
# =========================
def check_root_result(r):
    if r is None:
        return False, np.inf

    res = np.linalg.norm(r.fun)

    ok = (
        bool(r.success)
        and np.isfinite(res)
        and (res < minpack_error)
    )

    return ok, res


In [8]:
# =========================
# Initialize solving statistics and best solution
# =========================
# Optional: choose whether to provide Jacobian
use_shooting_jac = True

np.random.seed(322)

count_solved   = 0
total_time     = 0.0
total_nfev     = 0
converge_nfev  = 0

global_lambda  = None
global_m0      = -np.inf

# Record all converged max-m0 solutions
all_solutions = []

# =========================
# Random initial guess loop
# =========================

# =========================
# Common random initial guess pool
# =========================
np.random.seed(322)
initial_guess_pool = []
for i in range(repeat_num_minpack):
    lam0 = build_initial_guess_maxm0()
    initial_guess_pool.append(lam0.copy())
initial_guess_pool = np.asarray(initial_guess_pool)


In [9]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None

    # Random initial guess
    lam0 = initial_guess_pool[i].copy()

    time_start = time.perf_counter()
    r = run_root_once_maxm0(lam0, use_jac=use_shooting_jac)
    time_end = time.perf_counter()

    elapsed = time_end - time_start
    total_time += elapsed
    total_nfev += getattr(r, 'nfev', 0)

    ok, res = check_root_result(r)

    if ok:
        count_solved += 1
        converge_nfev += r.nfev

        m0_nu_sol = float(r.x[-1])  # direct m0_nu
        m0_si_sol = m0_nu_sol * MUnit

        print(
            f"counter: {count_solved}/{i+1}  nfev: {r.nfev}  res: {res:.3e}",
            end='',
            flush=True
        )

        # Print one more time with out=True
        shooting_maxm0(r.x, out=True, Time_out=None)

        # Since u = 1, mass decreases linearly in nondimensional variables.
        mf_nu_sol = m0_nu_sol - (c1 / c2) * (tf_nu - t0_nu)
        mf_si_sol = mf_nu_sol * MUnit

        # Save all converged solutions
        all_solutions.append({
            "trial_index": i,
            "x_init": lam0.copy(),
            "x_sol":  r.x.copy(),
            "m0_SI":  m0_si_sol,
            "mf_SI":  mf_si_sol,
            "res":    float(res),
            "nfev":   int(r.nfev),
            "time_s": float(elapsed),
        })

        # Save the best solution using m0
        if m0_si_sol > global_m0:
            global_m0 = m0_si_sol
            global_lambda = r.x.copy()
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence statistics
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), optimal m0: {global_m0:.6f} kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)
    print(f"Number of stored converged solutions: {len(all_solutions)}")

records_by_method['MIM_AJ'] = records


counter: 1/12  nfev: 61  res: 1.619e-12

  Final Mass: 1449.0859994614666   Initial Mass: 1684.0322086084584
counter: 2/13  nfev: 59  res: 4.362e-12

  Final Mass: 1449.08599946116   Initial Mass: 1684.0322086081537


counter: 3/22  nfev: 97  res: 4.038e-10

  Final Mass: 1378.3829954605994   Initial Mass: 1613.3292046075917


counter: 4/26  nfev: 128  res: 8.363e-12

... (2,445 outputs left out)

  Final Mass: 1449.0859994620555   Initial Mass: 1684.0322086090498


counter: 1365/4992  nfev: 119  res: 1.677e-09

  Final Mass: 1045.8496281733549   Initial Mass: 1280.7958373203496


Succeed to solve the problem!
Solved: 1365 in 5000 (0.2730), optimal m0: 1842.673245 kg
Average number of function calls for Minpack: 57
Average number of function calls for convergent solution: 94
global_lambda: [ 5.285643913593352  -4.378291652124971  -0.905317559203393
 -0.0037883370943241 -0.021263022854703  -0.0003081782375711
  1.2284488299102507]
Number of stored converged solutions: 1365


In [10]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(200)
accepted = np.array([r['accepted'] for r in records_by_method['MIM_AJ']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [11]:
%%timeit -n 1 -r 7 -v timing_all set_timing_priority()
for i in timing_ids:
    sample_cpu()

    # Random initial guess
    lam0 = initial_guess_pool[i].copy()

    r = run_root_once_maxm0(lam0, use_jac=use_shooting_jac)


1min 3s ± 584 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%%timeit -n 1 -r 7 -v timing_successful set_timing_priority()
for i in successful_ids:
    sample_cpu()

    # Random initial guess
    lam0 = initial_guess_pool[i].copy()

    r = run_root_once_maxm0(lam0, use_jac=use_shooting_jac)


7.65 s ± 34.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%%timeit -n 1 -r 7 -v timing_failed set_timing_priority()
for i in failed_ids:
    sample_cpu()

    # Random initial guess
    lam0 = initial_guess_pool[i].copy()

    r = run_root_once_maxm0(lam0, use_jac=use_shooting_jac)


55.2 s ± 563 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['MIM_AJ'] = dict(
    method='MIM, AJ', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['MIM_AJ'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['MIM_AJ']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Results


In [4]:
headings = ['Method', 'Expected first solution [ms]', 'Successful mean [ms]',
            'Failed mean [ms]', 'All-trial mean [ms]', 'Converged', 'NFEV, successful']
table = ['| ' + ' | '.join(headings) + ' |', '| ' + ' | '.join(['---'] * len(headings)) + ' |']
for row in results_by_method.values():
    values = [row['method']]
    values += [f"{row[key]:.3f}" if np.isfinite(row[key]) else '—'
               for key in ('first_ms', 'successful_ms', 'failed_ms', 'all_ms')]
    values += [f"{row['accepted']}/{row['trials']}", f"{row['nfev_successful']:.1f}"]
    table.append('| ' + ' | '.join(values) + ' |')
display(Markdown('\n'.join(table)))


| Method | Expected first solution [ms] | Successful mean [ms] | Failed mean [ms] | All-trial mean [ms] | Converged | NFEV, successful |
| --- | --- | --- | --- | --- | --- | --- |
| MINT, AJ | 1700.563 | 187.558 | 554.975 | 449.629 | 1322/5000 | 84.8 |
| MIM, AJ | 1163.666 | 144.281 | 375.661 | 317.681 | 1365/5000 | 94.0 |